## C2 - Silver Layer
- Handle nulls
- Standardize
- Remove duplicates

In [0]:
# Silver Layer – Data Cleaning & Transformation
# Performs data cleaning, standardization

from pyspark.sql.functions import col, upper

silver_df = spark.table("bronze_smartgear_orders")

# Handle nulls (drop rows with critical missing values)
silver_df = silver_df.dropna()

# Standardize region values
silver_df = silver_df.withColumn("region", upper(col("region")))

# Remove duplicates
silver_df = silver_df.dropDuplicates(["order_id", "timestamp"])

### Create Revenue
### Ensure Correct Data types

In [0]:
from pyspark.sql.functions import col

# Add revenue column and ensure correct data types
silver_df = (
    silver_df
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("revenue", col("quantity") * col("price"))  # Business metric
)

# Validate
silver_df.show(5)

+--------+--------------------+--------+----------+--------+------+------+-------+
|order_id|           timestamp|store_id|   product|quantity| price|region|revenue|
+--------+--------------------+--------+----------+--------+------+------+-------+
|    3475|2026-04-13 14:40:...|       6|    Laptop|       5|104.96|  WEST|  524.8|
|    4483|2026-04-13 14:40:...|       5|    Tablet|       5|377.18|  EAST| 1885.9|
|    6184|2026-04-13 14:41:...|       7|Headphones|       1|827.04|  EAST| 827.04|
|    1682|2026-04-13 14:42:...|       9|    Laptop|       1| 614.7| SOUTH|  614.7|
|    6146|2026-04-13 14:43:...|       3|Headphones|       2|758.98|  EAST|1517.96|
+--------+--------------------+--------+----------+--------+------+------+-------+
only showing top 5 rows


## Analytical Transformations
### Daily revenue trend (Aggregate revenue by date, Sort chronologically)

In [0]:
from pyspark.sql.functions import to_date, sum as _sum

# Extract date and calculate daily revenue
daily_revenue_df = (
    silver_df
    .withColumn("date", to_date(col("timestamp")))
    .groupBy("date")
    .agg(_sum("revenue").alias("daily_revenue"))
    .orderBy("date")
)

# Display result
display(daily_revenue_df)

date,daily_revenue
2025-12-14,172915.54000000004
2025-12-15,2.5988971210000023E7
2025-12-16,3.361577771E7
2025-12-17,3.473780542000001E7
2025-12-18,3.743380851999995E7
2025-12-19,3.449636625000005E7
2025-12-20,3.455341418000002E7
2025-12-21,3.861639650999999E7
2025-12-22,4.077338705000002E7
2025-12-23,3.3180016179999985E7


### Region-wise Contribution %  
-   % contribution of each region to total revenue

In [0]:
from pyspark.sql.functions import sum as _sum

# Total revenue
total_revenue = silver_df.agg(_sum("revenue")).collect()[0][0]

# Region-wise contribution
region_contribution_df = (
    silver_df
    .groupBy("region")
    .agg(_sum("revenue").alias("region_revenue"))
    .withColumn("contribution_pct", (col("region_revenue") / total_revenue) * 100)
)

display(region_contribution_df)

region,region_revenue,contribution_pct
SOUTH,945758.8699999996,25.682149719937406
NORTH,898293.1600000008,24.393214971925897
WEST,907155.34,24.633868103315525
EAST,931346.000000001,25.290767204821314


### Top 3 Stores per Region 
-  Use window functions 
-  Rank stores within each region

In [0]:
from pyspark.sql.functions import sum as _sum, col
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

# Step 1: Aggregate revenue per store per region
store_revenue_df = (
    silver_df
    .groupBy("region", "store_id")
    .agg(_sum("revenue").alias("total_revenue"))
)

# Step 2: Rank stores within each region
window_spec = Window.partitionBy("region").orderBy(col("total_revenue").desc())

top_stores_df = (
    store_revenue_df
    .withColumn("rank", rank().over(window_spec))
    .filter(col("rank") <= 3)
)

# Step 3: Display result
display(top_stores_df)

region,store_id,total_revenue,rank
EAST,6,2.3217930530000012E7,1
EAST,3,2.301818470999999E7,2
EAST,5,2.126941723000001E7,3
NORTH,9,2.1529138279999994E7,1
NORTH,2,2.040063887999999E7,2
NORTH,6,1.972554816999998E7,3
SOUTH,7,2.3758192850000024E7,1
SOUTH,6,2.091001549000001E7,2
SOUTH,9,2.0610733270000003E7,3
WEST,10,2.360213995E7,1


### Data Skew Identification 
- Identify if any region/store dominates data disproportionately

In [0]:
from pyspark.sql.functions import sum as _sum, col, round

# Step 1: Region-wise revenue
region_revenue_df = (
    silver_df
    .groupBy("region")
    .agg(_sum("revenue").alias("region_revenue"))
)

# Step 2: Total revenue
total_revenue = region_revenue_df.agg(_sum("region_revenue")).collect()[0][0]

# Step 3: Contribution %
region_skew_df = (
    region_revenue_df
    .withColumn("contribution_pct", round((col("region_revenue") / total_revenue) * 100, 2))
    .orderBy(col("contribution_pct").desc())
)

display(region_skew_df)

region,region_revenue,contribution_pct
EAST,1171646.7700000012,26.16
SOUTH,1143545.8399999996,25.53
NORTH,1091385.4800000007,24.36
WEST,1077907.939999999,24.06


The revenue distribution across regions is relatively uniform, with each region contributing approximately 24–26% of total revenue. This indicates that there is no significant data skew at the region level, and the dataset is well-balanced for downstream processing.

In [0]:
from pyspark.sql.functions import sum as _sum, col, round

# Step 1: Store-wise revenue
store_revenue_df = (
    silver_df
    .groupBy("store_id")
    .agg(_sum("revenue").alias("store_revenue"))
)

# Step 2: Total revenue
total_revenue = store_revenue_df.agg(_sum("store_revenue")).collect()[0][0]

# Step 3: Contribution %
store_skew_df = (
    store_revenue_df
    .withColumn("contribution_pct", round((col("store_revenue") / total_revenue) * 100, 2))
    .orderBy(col("contribution_pct").desc())
)

display(store_skew_df)

store_id,store_revenue,contribution_pct
7,8.062517238000007E7,10.42
6,8.040569441999988E7,10.4
10,7.950281054E7,10.28
3,7.888084780999984E7,10.2
8,7.828516154999991E7,10.12
5,7.70613299599999E7,9.96
9,7.673691237999997E7,9.92
2,7.539628026999982E7,9.75
1,7.517651599999985E7,9.72
4,7.143075546999995E7,9.23


Store-level revenue distribution is also balanced, with each store contributing approximately 9–11% of total revenue. No single store dominates the dataset, indicating absence of data skew at store level.

This confirms that both region-level and store-level distributions are uniform, ensuring unbiased downstream analytics.

### C3 – Gold Layer (Executive-ready analytics + advanced SQL)

In [0]:
from pyspark.sql.functions import sum as _sum

# Step 1: Region-wise revenue
region_revenue_df = (
    silver_df
    .groupBy("region")
    .agg(_sum("revenue").alias("region_revenue"))
)
display(region_revenue_df)

region,region_revenue
WEST,1.9338749918999997E8
EAST,1.9975757113999987E8
SOUTH,1.9887410554000008E8
NORTH,1.814823049100003E8


### Month-over-month growth (MoM)

In [0]:
from pyspark.sql.functions import to_date, date_format, sum as _sum

# Extract month from timestamp
monthly_revenue_df = (
    silver_df
    .withColumn("date", to_date(col("timestamp")))
    .withColumn("month", date_format(col("date"), "yyyy-MM"))
    .groupBy("month")
    .agg(_sum("revenue").alias("monthly_revenue"))
    .orderBy("month")
)

display(monthly_revenue_df)

month,monthly_revenue
2025-12,6.028484349200016E8
2026-01,1.3672733841999984E8
2026-02,1.0938368239999976E7
2026-03,1.245357866000002E7
2026-04,1.0533760539999947E7


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col, when, round

# Window for month ordering
window_spec = Window.orderBy(col("month"))

mom_df = (
    monthly_revenue_df
    .withColumn("prev_month_revenue", lag("monthly_revenue").over(window_spec))
    .withColumn(
        "mom_growth_pct",
        when(col("prev_month_revenue").isNull(), None)
        .otherwise(
            ((col("monthly_revenue") - col("prev_month_revenue")) / col("prev_month_revenue")) * 100
        )
    )
    .withColumn("mom_growth_pct", round(col("mom_growth_pct"), 2))
)

display(mom_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


month,monthly_revenue,prev_month_revenue,mom_growth_pct
2025-12,6.028484349200016E8,null,null
2026-01,1.3672733841999984E8,6.028484349200016E8,-77.32
2026-02,1.0938368239999976E7,1.3672733841999984E8,-92.0
2026-03,1.245357866000002E7,1.0938368239999976E7,13.85
2026-04,1.0533760539999947E7,1.245357866000002E7,-15.42


The first month does not have a previous reference, hence MoM growth is null. Subsequent months show positive or negative growth trends, indicating changes in business performance.

### Average order value per region

In [0]:
from pyspark.sql.functions import avg

aov_region_df = (
    silver_df
    .groupBy("region")
    .agg(avg("revenue").alias("avg_order_value"))
)

display(aov_region_df)

region,avg_order_value
WEST,20283.98355254877
EAST,20490.05755872396
SOUTH,20657.952169938722
NORTH,18902.437757525287


Calculated average revenue per order. Indicates customer spending behavior across regions

### Store performance segmentation: 
-  High / Medium / Low (based on revenue percentile)

In [0]:
from pyspark.sql.functions import when, col

# Get percentile thresholds
quantiles = store_revenue_df.approxQuantile("store_revenue", [0.33, 0.66], 0)

low_thresh = quantiles[0]
high_thresh = quantiles[1]

# Apply segmentation
store_segmentation_df = (
    store_revenue_df
    .withColumn(
        "segment",
        when(col("store_revenue") >= high_thresh, "High")
        .when(col("store_revenue") >= low_thresh, "Medium")
        .otherwise("Low")
    )
)

display(store_segmentation_df)

store_id,store_revenue,segment
6,8.040569441999988E7,High
5,7.70613299599999E7,Medium
7,8.062517238000007E7,High
9,7.673691237999997E7,Medium
3,7.888084780999984E7,High
8,7.828516154999991E7,Medium
2,7.539628026999982E7,Low
4,7.143075546999995E7,Low
10,7.950281054E7,High
1,7.517651599999985E7,Low


Stores were segmented into High, Medium, and Low categories using percentile-based thresholds (33rd and 66th percentiles). This ensures meaningful differentiation even when revenue distribution is relatively uniform.

## 📌 Solution Overview

The solution implements an end-to-end data pipeline using Kafka and Databricks to simulate real-time data ingestion, transformation, and analytics.

Data is generated using a Kafka producer simulating retail orders and consumed in Databricks for processing using a Medallion Architecture (Bronze, Silver, Gold layers).

---

## 📌 Data Ingestion (Kafka → Databricks)

* A Kafka producer was implemented to generate real-time order data including product, store, region, quantity, and price.
* The data was ingested into Databricks using Kafka connectors.
* Due to Databricks Serverless limitations, streaming was simulated using batch reads.

---

## 📌 Bronze Layer (Raw Data)

* Raw data from Kafka was ingested and converted from binary to string format.
* JSON data was parsed into structured columns.
* Metadata such as ingestion time and source system was added.
* This layer retains raw data with minimal transformation.

---

## 📌 Silver Layer (Data Cleaning & Transformation)

* Removed null and duplicate records to ensure data quality.
* Standardized categorical fields such as region.
* Derived revenue metric (quantity × price).
* Aggregated data for:

  * Daily revenue trends
  * Region-wise revenue contribution
  * Top-performing stores using window functions
* Performed data skew analysis at both region and store levels, confirming balanced distribution.

---

## 📌 Gold Layer (Business KPIs & Insights)

* Calculated Monthly Revenue for trend analysis.
* Derived Month-over-Month (MoM) growth using window functions.
* Computed Average Order Value (AOV) to measure customer spending behavior across regions
* Implemented store segmentation using percentile-based thresholds to classify stores into High, Medium, and Low performers.

---

## 📌 Key Insights

* Revenue distribution across regions and stores was balanced, indicating no significant data skew.
* Top-performing stores were identified per region based on aggregated revenue.
* MoM growth provided visibility into business performance trends.
* Store segmentation enabled classification of performance tiers for targeted decision-making.

---

## 📌 Limitations

* Databricks Serverless environment does not support continuous streaming (writeStream), so batch processing was used to simulate streaming.
* Initial dataset contained single-month data; this was resolved by generating multi-month data for meaningful MoM analysis.

---

## 📌 Conclusion

The solution demonstrates a scalable data pipeline architecture with real-time ingestion, transformation, and analytics. It effectively showcases the use of distributed data processing, window functions, and business KPI generation, aligning with real-world data engineering and analytics practices.


## Advanced SQL Analytics

### Top 5 Products by Revenue (Global + Region-wise) 
-  Use RANK() / DENSE_RANK()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum as _sum, col, rank

# Global Top 5 Products
product_global_df = (
    silver_df
    .groupBy("product")
    .agg(_sum("revenue").alias("total_revenue"))
)

window_global = Window.orderBy(col("total_revenue").desc())

top_products_global = (
    product_global_df
    .withColumn("rank", rank().over(window_global))
    .filter(col("rank") <= 5)
)

display(top_products_global)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


product,total_revenue,rank
Laptop,3.546691728500012E8,1
Phone,2.0712466803999993E8,2
Tablet,1.7368365696000007E8,3
Headphones,3.802398292999994E7,4


####  Region-wise Top Products

In [0]:
# Region-wise Top 5 Products
product_region_df = (
    silver_df
    .groupBy("region", "product")
    .agg(_sum("revenue").alias("total_revenue"))
)

window_region = Window.partitionBy("region").orderBy(col("total_revenue").desc())

top_products_region = (
    product_region_df
    .withColumn("rank", rank().over(window_region))
    .filter(col("rank") <= 5)
)

display(top_products_region)

region,product,total_revenue,rank
EAST,Laptop,9.099083780999984E7,1
EAST,Phone,5.469019618999986E7,2
EAST,Tablet,4.477622598999995E7,3
EAST,Headphones,9300311.149999995,4
NORTH,Laptop,8.41370040799999E7,1
NORTH,Phone,4.7663835400000006E7,2
NORTH,Tablet,3.988878895999996E7,3
NORTH,Headphones,9792676.469999997,4
SOUTH,Laptop,9.111865416000018E7,1
SOUTH,Phone,5.4715074679999985E7,2


Identified top-performing products globally and per region using ranking window functions. This helps in product-level performance analysis and regional demand insights.

### Rolling 7-Day Revenue (Moving Average)
-  Use window frame

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

# Ensure daily revenue exists
window_spec = Window.orderBy("date").rowsBetween(-6, 0)

rolling_df = (
    daily_revenue_df
    .withColumn("rolling_avg_7d", avg("daily_revenue").over(window_spec))
)

display(rolling_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


date,daily_revenue,rolling_avg_7d
2025-12-14,172915.54000000004,172915.54000000004
2025-12-15,2.5988971210000023E7,1.3080943375000011E7
2025-12-16,3.361577771E7,1.992588815333334E7
2025-12-17,3.473780542000001E7,2.3628867470000006E7
2025-12-18,3.743380851999995E7,2.6389855679999996E7
2025-12-19,3.449636625000005E7,2.7740940775000006E7
2025-12-20,3.455341418000002E7,2.8714151261428576E7
2025-12-21,3.861639650999999E7,3.420607711428572E7
2025-12-22,4.077338705000002E7,3.631813652E7
2025-12-23,3.3180016179999985E7,3.6255884872857146E7


Computed 7-day rolling average revenue using window frame. This smooths short-term fluctuations and highlights underlying trends.

### Anomaly Detection (Basic) 
-  Identify days where revenue deviates >30% from average

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, col

# Step 1: Rolling average (reuse if already created)
window_spec = Window.orderBy("date").rowsBetween(-3, 0)

rolling_df = (
    daily_revenue_df
    .withColumn("rolling_avg", avg("daily_revenue").over(window_spec))
)

# Step 2: Deviation from rolling average
anomaly_df = (
    rolling_df
    .withColumn(
        "deviation_pct",
        ((col("daily_revenue") - col("rolling_avg")) / col("rolling_avg")) * 100
    )
    .filter(col("deviation_pct") > 30)
)

display(anomaly_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


date,daily_revenue,rolling_avg,deviation_pct
2025-12-15,2.5988971210000023E7,1.3080943375000011E7,98.678111088452
2025-12-16,3.361577771E7,1.992588815333334E7,68.7040369358719
2025-12-17,3.473780542000001E7,2.3628867470000006E7,47.014263227403
2026-04-08,629810.8200000001,443422.38499999995,42.03406081991105
2026-04-13,4494055.850000003,1600199.0800000005,180.8435466667061


Identified anomalous days where revenue deviates more than 30% from average. This helps detect unusual spikes or drops in business performance.

### Customer Behavior Proxy (Optional Challenge Bonus)
-  (since no customer ID — force thinking)
-   Use StoreID as proxy → identify “repeat behavior patterns

In [0]:
# ---------------------------------------------
# Customer Behavior Proxy using Store ID
# ---------------------------------------------
# Since customer_id is not available, store_id is used as a proxy
# to analyze repeat behavior patterns (frequency, value, consistency).
# ---------------------------------------------

from pyspark.sql.functions import count, sum, to_date, countDistinct, col, when

# 1. Order frequency (repeat behavior)
store_frequency = (
    silver_df
    .groupBy("store_id")
    .agg(count("order_id").alias("total_orders"))
)

# 2. Revenue contribution (value behavior)
store_revenue = (
    silver_df
    .groupBy("store_id")
    .agg(sum("revenue").alias("total_revenue"))
)

# 3. Active days (consistency behavior)
store_activity = (
    silver_df
    .withColumn("date", to_date("timestamp"))
    .groupBy("store_id")
    .agg(countDistinct("date").alias("active_days"))
)

# 4. Combine all metrics
store_behavior = (
    store_frequency
    .join(store_revenue, "store_id")
    .join(store_activity, "store_id")
)

# 5. Dynamic segmentation using quantiles (ensures meaningful grouping)
quantiles = store_behavior.approxQuantile("total_orders", [0.33, 0.66], 0)

low_threshold = quantiles[0]
mid_threshold = quantiles[1]

store_behavior = store_behavior.withColumn(
    "store_type",
    when(col("total_orders") > mid_threshold, "High")
    .when(col("total_orders") > low_threshold, "Medium")
    .otherwise("Low")
)

# 6. Final output
display(store_behavior)

store_id,total_orders,total_revenue,active_days,store_type
10,3833,7.950281054E7,122,Medium
5,3839,7.70613299599999E7,122,Medium
3,3819,7.888084780999984E7,122,Low
1,3968,7.517651599999985E7,122,High
2,3821,7.539628026999982E7,122,Medium
9,3896,7.673691237999997E7,122,High
6,3907,8.040569441999988E7,122,High
7,3806,8.062517238000007E7,122,Low
4,3812,7.143075546999995E7,122,Low
8,3810,7.828516154999991E7,122,Low


## Insights (Business View)
 - Month-over-Month (MoM) trends highlight growth/decline periods, helping identify seasonality.
 - Regional AOV differences indicate where higher-value orders are concentrated.
  - Store segmentation (High/Medium/Low) identifies top-performing vs underperforming stores.
- These insights can guide inventory planning, regional targeting, and store-level strategies.